# 랭체인 에이전트

- https://python.langchain.com/docs/tutorials/agents/#installation

- https://python.langchain.com/docs/integrations/tools/

### 빌트인 도구
- 랭체인에서 제공하는 사전에 정의된 도구와 툴킷
- 툴은 단일 도구, 툴킷은 여러 도구를 묶어서 하나의 도구로 활용 가능
- Langchain Tools/toolkit


## 1. 검색을 하는 툴 사용해보기

### Tavily

- 대형 언어 모델(LLM)과 AI 에이전트를 위해 설계된 고성능 검색 엔진이자 API 플랫폼
- AI가 실시간으로 정확하고 사실에 기반한 정보를 검색하고 활용할 수 있도록 돕는 도구
- https://www.tavily.com/

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
# %pip install -qU langchain langchain-openai langchain-tavily tiktoken langchain-community

# uv add langchain langchain-openai langchain-tavily tiktoken langchain-community

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# 웹 서치 툴 : 최신 뉴스/웹 결과
from langchain_tavily import TavilySearch


## 2. 파일 저장하는 툴 사용해보기

In [ ]:
# agent_filesystem_news.py
import os, datetime, csv
from pathlib import Path
from typing import List, Dict

In [ ]:
# 파일 툴 정의에 사용할 LangChain v1 tool 데코레이터
from langchain.tools import tool

In [ ]:
# --- 샌드박스 디렉터리(여기 밖은 접근 불가) ---
ROOT = Path("../01_agent_result").absolute()
ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 이전 버전에서는 이렇게 사용
# # 파일 툴킷 : 읽기/쓰기/목록
# from langchain_community.agent_toolkits import FileManagementToolkit\

# # 파일 관리 툴킷 (read_file / write_file / list_directory 등 제공)
# file_tools = FileManagementToolkit(root_dir=str(ROOT)).get_tools()
# # 참고: 이 툴킷은 루트 디렉터리 내에서 Copy/Move/Delete/Read/Write/List 등을 제공. :contentReference[oaicite:1]{index=1}

In [ ]:
# 파일 관리 도구 직접 정의 (read_file / write_file / list_directory)
def _safe_path(path: str) -> Path:
    """ROOT 밖으로 나가지 않도록 경로를 검증합니다."""
    target = (ROOT / (path or ".")).resolve()
    root = ROOT.resolve()
    if root != target and root not in target.parents:
        raise ValueError(f"ROOT 밖의 경로는 사용할 수 없습니다: {path}")
    return target

@tool
def read_file(file_path: str) -> str:
    """Read a UTF-8 text file under the sandbox root directory."""
    return _safe_path(file_path).read_text(encoding="utf-8")

@tool
def write_file(file_path: str, text: str) -> str:
    """Write UTF-8 text to a file under the sandbox root directory."""
    target = _safe_path(file_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding="utf-8")
    return f"saved: {target.relative_to(ROOT.resolve()).as_posix()}"

@tool
def list_directory(directory_path: str = ".") -> str:
    """List files and directories under the sandbox root directory."""
    target = _safe_path(directory_path)
    if not target.exists():
        return f"not found: {directory_path}"
    if target.is_file():
        return target.relative_to(ROOT.resolve()).as_posix()
    entries = []
    for item in sorted(target.iterdir(), key=lambda p: p.name):
        rel = item.relative_to(ROOT.resolve()).as_posix()
        entries.append(rel + ("/" if item.is_dir() else ""))
    return "\n".join(entries) if entries else "(empty)"

file_tools = 


In [ ]:
# 뉴스/웹 검색 툴
tavily =

In [ ]:
# --- 프롬프트 ---
SYSTEM = """
너는 '파일 시스템 툴'과 '웹 검색 툴'만 사용하는 어시스턴트다.
규칙
1) 최신 뉴스/링크는 Tavily로 검색한다.
2) 사용자가 CSV 저장을 요청하면, 다음 필드 헤더로 CSV를 만든다.:
    data, query, title, url, source, snippet
3) 파일 경로는 사용자가 지정하지 않으면 01_agent_result/new_{YYYYMMDD}.csv 로 저장한다.
4) 같은 날짜 파일이 이미 있더라도 새로 '덮어쓰기'하지 말고, 기존 내용을 읽어 헤더를 유지한 채 뒤로 이어붙인 텍스트를 만들고, write_file로 한번에 저장한다.
5) 저장 뒤에는 list_directory 또는 read_file을 활용해 저장 확인(행수/상위 3~5행) 정보를 보여준다.
6) 샌드박스 디렉터리(Root) 밖의 경로는 절대 사용하지 않는다.
모든 답변은 한국어로 간결하게.
"""

llm = 
TOOLS = 

agent = 

In [ ]:
from datetime import datetime
def get_today():
    return datetime.now().strftime("%Y%m%d")

In [ ]:
print("\n=== A. 뉴스 요약 ===")
user_input = f"{get_today()} 오늘 한국 AI 반도체 관련 최신 뉴스 5개만 핵심 요약해줘."
result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
print(result["messages"][-1].content)


In [ ]:
print("\n=== B. 검색 후 CSV 저장 ===")
user_input = f"{get_today()} 한국 반도체 최신 뉴스 5개를 찾아서 csv로 저장하고, 저장된 파일 상위 5행도 보여줘."
result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
print(result["messages"][-1].content)


In [ ]:
print("\n=== C. 작업 디렉터리 목록 ===")
user_input = f"{get_today()} 내 작업 폴더 안 파일 목록을 보여줘"
result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
print(result["messages"][-1].content)


In [ ]:
# C) 디렉터리 내 파일 목록 보기
print("\n=== C. 작업 디렉터리 목록 ===")
user_input = "내 작업 폴더 안 파일 목록을 보여줘"
result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
print(result["messages"][-1].content)


## 3. 웹검색 -> sqlite 저장 -> 확인까지 하는 툴 만들기

- SQLite : 설치 없이 사용할 수 있는 가벼운 파일 기반 데이터베이스
- 특징 요약
    - 서버가 필요 없음 (파일 1개로 DB 완성)
    - SQL 문법 사용 (표준 SQL 지원)
    - 속도가 빠르고 가볍다
    - 파이썬 등 언어에 내장되어 있음 (sqlite3 모듈)

In [ ]:
import os, datetime, sqlite3
from pathlib import Path
from langchain.tools import tool
from langchain_tavily import TavilySearch


In [ ]:
# --- 작업 디렉터리 & SQLite ---
ROOT = Path("../01_agent_result");
ROOT.mkdir(exist_ok=True, parents=True)
DB_URI = f"sqlite:///{(ROOT / 'news.db').as_posix()}"

In [ ]:
def _connect_db():
    return sqlite3.connect(DB_PATH)

In [ ]:
# --- 툴 구성: SQL + Tavily ---
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def _ensure_news_table() -> None:
    with _connect_db() as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS news (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                date TEXT NOT NULL,
                query TEXT NOT NULL,
                title TEXT NOT NULL,
                url TEXT NOT NULL UNIQUE,
                source TEXT,
                snippet TEXT
            )
            """
        )
        conn.commit()

@tool
def save_news_item(date: str, query: str, title: str, url: str, source: str = "", snippet: str = "") -> str:
    """Save one news item to the local SQLite news table using safe parameter binding."""
    _ensure_news_table()
    with _connect_db() as conn:
        cursor = conn.execute(
            """
            INSERT OR IGNORE INTO news (date, query, title, url, source, snippet)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (date, query, title, url, source or "", snippet or ""),
        )
        conn.commit()
    if cursor.rowcount == 0:
        return f"duplicate skipped: {url}"
    return f"saved: {title}"

@tool
def list_tables() -> str:
    """List table names in the SQLite database."""
    _ensure_news_table()
    with _connect_db() as conn:
        rows = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        ).fetchall()
    return "\n".join(row[0] for row in rows) if rows else "(no tables)"

@tool
def describe_table(table_name: str) -> str:
    """Show columns and types for a SQLite table."""
    _ensure_news_table()
    with _connect_db() as conn:
        rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    if not rows:
        return f"table not found: {table_name}"
    return "\n".join(f"{row[1]} {row[2]}" for row in rows)

@tool
def run_sql(query: str) -> str:
    """Run SELECT/PRAGMA/WITH queries against the local SQLite database and return results."""
    _ensure_news_table()
    statement = query.strip().rstrip(";")
    if not statement:
        return "empty query"

    lowered = statement.lstrip().lower()
    if not lowered.startswith(("select", "pragma", "with")):
        return "run_sql은 조회 전용입니다. 저장은 save_news_item 도구를 사용하세요."

    try:
        with _connect_db() as conn:
            cursor = conn.execute(statement)
            rows = cursor.fetchall()
            columns = [desc[0] for desc in cursor.description or []]
    except sqlite3.Error as exc:
        return f"SQL error: {exc}"

    outputs = []
    if columns:
        outputs.append(" | ".join(columns))
    outputs.extend(" | ".join(str(value) for value in row) for row in rows)
    return "\n".join(outputs) if outputs else "(no rows)"

tavily = TavilySearch(max_results=5)
sql_tools = [save_news_item, list_tables, describe_table, run_sql]
Tools = [tavily] + sql_tools


In [ ]:
# --- 프롬프트 ---
SYSTEM = """너는 '웹검색'과 'SQL 데이터베이스' 툴만 사용한다.
목표: 사용자가 원하면 최신 뉴스를 검색해서 SQLite(news.db)에 저장하고, 요청 시 조회/요약까지 수행한다.

[테이블 스키마]
테이블명: news
컬럼: id INTEGER PRIMARY KEY AUTOINCREMENT,
       date TEXT NOT NULL,         -- YYYY-MM-DD
       query TEXT NOT NULL,        -- 사용자가 요청한 검색어/명령
       title TEXT NOT NULL,
       url TEXT NOT NULL UNIQUE,   -- 중복 방지
       source TEXT,
       snippet TEXT

규칙:
1) 저장 전, news 테이블이 없는 경우 SQL로 직접 CREATE TABLE IF NOT EXISTS를 실행한다.
2) 웹 검색은 Tavily 툴로 수행하고 결과를 title/url/source/snippet으로 정규화한다.
3) date=오늘(한국시간 기준 YYYY-MM-DD), query=사용자 질의를 넣어 INSERT OR IGNORE로 저장한다.
4) 저장 후에는 방금 저장한 레코드 수와 상위 5건을 SELECT로 보여준다.
5) 조회만 요청하면 SELECT만 수행한다.
6) 항상 안전하고 간결한 SQL을 작성하고, 불필요한 전체 스캔/전체 컬럼 조회를 피한다.
7) 모든 답변은 한국어로 간결하게. 마지막에 한 줄 결론을 덧붙여라.
"""


In [ ]:
today = datetime.date.today().isoformat()
q = f"{today} 한국 AI 반도체 최신 뉴스 5개를 찾아 DB에 저장하고 상위 5건만 보여줘."
print("\n=== A. 검색 후 저장 & 확인 ===")
result = agent.invoke({"messages": [{"role": "user", "content": q}]})
print(result["messages"][-1].content)


In [ ]:
# B) 조회만 (예: 오늘 저장분 5건)
q2 = "오늘 저장된 뉴스 중 제목과 출처만 5건 보여줘"
print("\n=== B. 조회만 ===")
result = agent.invoke({"messages": [{"role": "user", "content": q2}]})
print(result["messages"][-1].content)
